[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/spinors/spinors.ipynb)

# Dirac, Weyl and Majorana spinors

A spinor turns differently from the geometry it carries: one complete rotation restores its spin direction but reverses its sign. Interference with a fixed reference sees that sign; a second rotation restores both. Start with that experiment, where the arrow traces the spin direction in the xz plane and the two disks show the intensity at the sum and difference ports; then use maps on the same eight-real-component spinor space to select Weyl and Majorana states.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline

import numpy as np

from numga import NumpyContext
from numga.algebras import STA
from examples.animation import save_figure, show_animation
from examples.relativity.spinors import render

In [ ]:
# Spacetime has one positive time direction and three negative space directions.
ga = STA
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Spinor = ga.gatype.even()


def density(psi: Spinor) -> Scalar:
    """The probability density the fixed time observer measures: the time part of the current."""
    return (psi >> mv.t) | mv.t

In [ ]:
frames = 96
duration_ms = 80
angles = np.linspace(0, 4 * np.pi, frames, endpoint=False)

reference = mv.scalar()
psi = (mv.xz * (angles / 2)).exp()                           # [frames] Spinor
# Left action turns the spinor; its sandwich map carries the observable geometry.
frame = psi >> Vector                                        # [frames] Vector <- Vector
spin = frame(mv.z)                                           # [frames] Vector
# Recombine with the reference in two ports. The time current gives each intensity.
bright = density((psi + reference) / 2)                      # [frames] Scalar
dark = density((psi - reference) / 2)                        # [frames] Scalar
show_animation(render.animate_rotation(spin, bright, dark), "spinor_rotation", duration_ms)

## Weyl: select a handedness

Right multiplication by `mv.zt` squares to the identity. Its two projectors split an eight-real-component Dirac spinor into right and left Weyl parts, each with four real degrees of freedom. These are constraints within `Spinor`, not selections of blade grades. Each part has a lightlike current; a boost changes their relative weights, the two currents add to the full Dirac current `psi >> mv.t`, and their difference is the spin current `psi >> mv.z`.

In Dirac-matrix notation the two projectors read as $\tfrac12(1\pm\gamma^5)\psi$.

In [ ]:
rapidity = 0.7

chirality = Spinor * mv.zt                                    # [] Spinor <- Spinor
right_projector = (Spinor + chirality) / 2
left_projector = (Spinor - chirality) / 2
psi = (mv.tz * (-rapidity / 2)).exp()
right = right_projector(psi)                                 # [] Spinor
left = left_projector(psi)                                   # [] Spinor
# Each Weyl current is null; the two add to the Dirac current.
right_current = right >> mv.t                                # [] Vector
left_current = left >> mv.t                                  # [] Vector
current = psi >> mv.t                                        # [] Vector
save_figure(render.draw_chirality(left_current, right_current, current), "spinor_chirality");

## Majorana: select a reality condition

Real coefficients alone do not make a spinor Majorana: charge conjugation must leave it unchanged. Right multiplication by `mv.yt` imposes that condition on another four-real-dimensional part of `Spinor`. Charge conjugation reverses a phase turn generated by `mv.yx`, whereas chirality commutes with it. Any phase preserves a Weyl state; only phases zero and half a turn preserve this Majorana condition. The curves show density moving between the two conjugation eigenspaces.

In Dirac-matrix notation the Majorana condition reads as $\psi = \psi_c$.

In [ ]:
phase_samples = 129
phase_angles = np.linspace(0, 2 * np.pi, phase_samples)

# Charge conjugation squares to the identity, so half its sum with the identity is a projector.
charge_conjugation = Spinor * mv.yt                           # [] Spinor <- Spinor
majorana_projector = (Spinor + charge_conjugation) / 2
majorana = majorana_projector(mv.scalar())
majorana = majorana / density(majorana).square_root()
phase = (mv.yx * phase_angles).exp()
phased = majorana * phase                                    # [phase_samples] Spinor
conjugated = charge_conjugation(phased)
# The parts that conjugation fixes and reverses.
fixed = (phased + conjugated) / 2                            # [phase_samples] Spinor
negated = (phased - conjugated) / 2                          # [phase_samples] Spinor
fixed_density = density(fixed)                               # [phase_samples] Scalar
negated_density = density(negated)                           # [phase_samples] Scalar
save_figure(render.draw_majorana(phase_angles, fixed_density, negated_density), "spinor_majorana");

Phase, chirality and charge conjugation are all `Spinor <- Spinor` extensors. The distinction between Dirac, Weyl and Majorana states lies in which maps constrain the state. The observable current is then read from its action on the time direction.

The spacetime-algebra Weyl and Majorana constructions follow [Francis and Kosowsky, *The Construction of Spinors in Geometric Algebra*, §VI.C](https://arxiv.org/pdf/math-ph/0403040).